In [ ]:
import os
import ast
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import joblib

# =========================
# 1. PATH & LOAD DATA
# =========================

train_path = r"C:\Fauzan\Manuskrip QSAR 3\Revisi 1\Endpoint\Developmental Toxicity\Dev_Train set_with_fingerprints_RDKit_CDK.xlsx"
out_dir    = r"C:\Fauzan\Manuskrip QSAR 3\Revisi 1\Endpoint\Developmental Toxicity\DL Models"

os.makedirs(out_dir, exist_ok=True)

train_df = pd.read_excel(train_path)

physchem_cols = [
    "MolWt","logP","LabuteASA","TPSA","AMW",
    "NumRotatableBonds","NumAromaticRings","NumSaturatedRings","NumAliphaticRings",
    "NumAromaticHeterocycles","NumSaturatedHeterocycles","NumAliphaticHeterocycles",
    "NumAromaticCarbocycles","NumSaturatedCarbocycles","NumAliphaticCarbocycles",
    "FractionCSP3","Chi0v","Chi1v","Chi2v","Chi3v","Chi4v",
    "Chi1n","Chi2n","Chi3n","Chi4n","HallKierAlpha",
    "HeavyAtomCount","RingCount","NumHDonors","NumHAcceptors",
    "ALogP","ALogp2","AMR","MLogP","nAtomP","naAromAtom","bpol","nB",
    "ECCEN","fragC","nHBAcc","nHBDon","nAtomLAC","nAtomLC",
    "PetitjeanNumber","nRotB","LipinskiFailures","TopoPSA","VAdjMat","XLogP","Fsp3"
]
target_col = "Outcome"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# scaler physicochemical (fit di seluruh train, simpan)
scaler = StandardScaler()
scaler.fit(train_df[physchem_cols].values.astype("float32"))
scaler_path = os.path.join(out_dir, "physicochemical_scaler.joblib")
joblib.dump(scaler, scaler_path)
print("Scaler saved to:", scaler_path)

# =================================================
# 2. DATASET, MODEL, DAN METRIC
# =================================================

class FingerprintOnlyDataset(Dataset):
    def __init__(self, df, fp_col, target_col):
        self.fp = df[fp_col].apply(ast.literal_eval).tolist()
        self.fp = torch.tensor(self.fp, dtype=torch.float32)
        self.y = torch.tensor(df[target_col].values.astype("float32"))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.fp[idx], self.y[idx]

class PhyschemOnlyDataset(Dataset):
    def __init__(self, df, physchem_cols, scaler, target_col):
        X = df[physchem_cols].values.astype("float32")
        X = scaler.transform(X)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(df[target_col].values.astype("float32"))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class FPModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

class PhyschemModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)

def get_input_dim_from_col(df, fp_col):
    example = ast.literal_eval(df[fp_col].iloc[0])
    return len(example)

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    acc = (TP + TN) / max((TP + TN + FP + FN), 1)
    sen = TP / max((TP + FN), 1)
    spe = TN / max((TN + FP), 1)
    bacc = (sen + spe) / 2.0
    return {"AUC": auc, "ACC": acc, "SEN": sen, "SPE": spe, "BACC": bacc}

# =================================================
# 3. 10-FOLD CV UNTUK FINGERPRINT & PHYSCHEM
# =================================================

def run_10fold_cv_fp(df, fp_col, name,
                     target_col="Outcome",
                     n_splits=10, batch_size=64,
                     n_epochs=80, lr=1e-3, seed=42):
    X = df[fp_col].values
    y = df[target_col].values
    input_dim = get_input_dim_from_col(df, fp_col)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    metrics_list = []

    fold_idx = 1
    for train_idx, val_idx in skf.split(X, y):
        print(f"\n[{name}] Fold {fold_idx}/{n_splits}")
        fold_idx += 1
        train_df_fold = df.iloc[train_idx].reset_index(drop=True)
        val_df_fold   = df.iloc[val_idx].reset_index(drop=True)

        train_ds = FingerprintOnlyDataset(train_df_fold, fp_col, target_col)
        val_ds   = FingerprintOnlyDataset(val_df_fold,   fp_col, target_col)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

        y_train = train_df_fold[target_col].values
        pos = (y_train == 1).sum()
        neg = (y_train == 0).sum()
        pos_weight = max(neg / max(pos, 1), 1.0)

        model = FPModel(input_dim).to(device)
        pos_weight_t = torch.tensor([pos_weight], dtype=torch.float32, device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        for epoch in range(1, n_epochs+1):
            model.train()
            total_loss = 0.0
            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                optimizer.zero_grad()
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * y_batch.size(0)
            if epoch == 1 or epoch % 10 == 0:
                avg_loss = total_loss / len(train_loader.dataset)
                print(f"  Epoch {epoch:03d} | train loss {avg_loss:.4f}")

        model.eval()
        all_probs = []
        all_true = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                logits = model(X_batch)
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs)
                all_true.append(y_batch.numpy())
        all_probs = np.concatenate(all_probs)
        all_true = np.concatenate(all_true)
        fold_metrics = compute_metrics(all_true, all_probs, threshold=0.5)
        print(f"  Fold metrics: "
              f"AUC={fold_metrics['AUC']:.4f}, "
              f"ACC={fold_metrics['ACC']:.4f}, "
              f"BACC={fold_metrics['BACC']:.4f}, "
              f"SEN={fold_metrics['SEN']:.4f}, "
              f"SPE={fold_metrics['SPE']:.4f}")
        metrics_list.append(fold_metrics)

    keys = ["AUC", "ACC", "BACC", "SEN", "SPE"]
    mean_metrics = {k: np.mean([m[k] for m in metrics_list]) for k in keys}
    std_metrics  = {k: np.std([m[k] for m in metrics_list])  for k in keys}

    print(f"\n=== {name} 10-fold CV results ===")
    for k in keys:
        print(f"{k}: {mean_metrics[k]:.4f} ± {std_metrics[k]:.4f}")

    df_folds = pd.DataFrame(metrics_list)
    df_folds.to_excel(os.path.join(out_dir, f"{name}_10fold_results.xlsx"), index=False)
    df_summary = pd.DataFrame({
        "Metric": list(mean_metrics.keys()),
        "Mean": [mean_metrics[k] for k in mean_metrics],
        "Std":  [std_metrics[k]  for k in std_metrics],
    })
    df_summary.to_excel(os.path.join(out_dir, f"{name}_10fold_summary.xlsx"), index=False)
    print(f"{name} CV results saved to {out_dir}")

    return mean_metrics, std_metrics, metrics_list

def run_10fold_cv_physchem(df, name,
                           physchem_cols, scaler,
                           target_col="Outcome",
                           n_splits=10, batch_size=64,
                           n_epochs=80, lr=1e-3, seed=42):
    X = df[physchem_cols].values
    y = df[target_col].values
    input_dim = len(physchem_cols)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    metrics_list = []

    fold_idx = 1
    for train_idx, val_idx in skf.split(X, y):
        print(f"\n[{name}] Fold {fold_idx}/{n_splits}")
        fold_idx += 1
        train_df_fold = df.iloc[train_idx].reset_index(drop=True)
        val_df_fold   = df.iloc[val_idx].reset_index(drop=True)

        train_ds = PhyschemOnlyDataset(train_df_fold, physchem_cols, scaler, target_col)
        val_ds   = PhyschemOnlyDataset(val_df_fold,   physchem_cols, scaler, target_col)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

        y_train = train_df_fold[target_col].values
        pos = (y_train == 1).sum()
        neg = (y_train == 0).sum()
        pos_weight = max(neg / max(pos, 1), 1.0)

        model = PhyschemModel(input_dim).to(device)
        pos_weight_t = torch.tensor([pos_weight], dtype=torch.float32, device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        for epoch in range(1, n_epochs+1):
            model.train()
            total_loss = 0.0
            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                optimizer.zero_grad()
                logits = model(X_batch)
                loss = criterion(logits, y_batch)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * y_batch.size(0)
            if epoch == 1 or epoch % 10 == 0:
                avg_loss = total_loss / len(train_loader.dataset)
                print(f"  Epoch {epoch:03d} | train loss {avg_loss:.4f}")

        model.eval()
        all_probs = []
        all_true = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device)
                logits = model(X_batch)
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs)
                all_true.append(y_batch.numpy())
        all_probs = np.concatenate(all_probs)
        all_true = np.concatenate(all_true)
        fold_metrics = compute_metrics(all_true, all_probs, threshold=0.5)
        print(f"  Fold metrics: "
              f"AUC={fold_metrics['AUC']:.4f}, "
              f"ACC={fold_metrics['ACC']:.4f}, "
              f"BACC={fold_metrics['BACC']:.4f}, "
              f"SEN={fold_metrics['SEN']:.4f}, "
              f"SPE={fold_metrics['SPE']:.4f}")
        metrics_list.append(fold_metrics)

    keys = ["AUC", "ACC", "BACC", "SEN", "SPE"]
    mean_metrics = {k: np.mean([m[k] for m in metrics_list]) for k in keys}
    std_metrics  = {k: np.std([m[k] for m in metrics_list])  for k in keys}

    print(f"\n=== {name} 10-fold CV results ===")
    for k in keys:
        print(f"{k}: {mean_metrics[k]:.4f} ± {std_metrics[k]:.4f}")

    df_folds = pd.DataFrame(metrics_list)
    df_folds.to_excel(os.path.join(out_dir, f"{name}_10fold_results.xlsx"), index=False)
    df_summary = pd.DataFrame({
        "Metric": list(mean_metrics.keys()),
        "Mean": [mean_metrics[k] for k in mean_metrics],
        "Std":  [std_metrics[k]  for k in std_metrics],
    })
    df_summary.to_excel(os.path.join(out_dir, f"{name}_10fold_summary.xlsx"), index=False)
    print(f"{name} CV results saved to {out_dir}")

    return mean_metrics, std_metrics, metrics_list

# =================================================
# 4. JALANKAN SEMUA: MORGAN, MACCS, APF, PHYSCHEM
# =================================================

morgan_mean, morgan_std, morgan_folds = run_10fold_cv_fp(
    df=train_df,
    fp_col="Morgan_Descriptors",
    name="Morgan",
    target_col=target_col,
    n_splits=10,
    batch_size=64,
    n_epochs=80,
    lr=1e-3,
    seed=42
)

maccs_mean, maccs_std, maccs_folds = run_10fold_cv_fp(
    df=train_df,
    fp_col="MACCS_Descriptors",
    name="MACCS",
    target_col=target_col,
    n_splits=10,
    batch_size=64,
    n_epochs=80,
    lr=1e-3,
    seed=42
)

apf_mean, apf_std, apf_folds = run_10fold_cv_fp(
    df=train_df,
    fp_col="APF_Descriptors",
    name="APF",
    target_col=target_col,
    n_splits=10,
    batch_size=64,
    n_epochs=80,
    lr=1e-3,
    seed=42
)

phys_mean, phys_std, phys_folds = run_10fold_cv_physchem(
    df=train_df,
    name="Physicochemical",
    physchem_cols=physchem_cols,
    scaler=scaler,
    target_col=target_col,
    n_splits=10,
    batch_size=64,
    n_epochs=80,
    lr=1e-3,
    seed=42
)

print("All 10-fold CV finished.")
